# ============================================================
# MEDICAL / TREATMENT DATASET PIPELINE V3
# Resume-safe + checkpointed + Excel master index
#
# Outputs:
# /kaggle/working/prepared_medical_dataset_v3/
#   manifests/
#   reports/
#   checkpoints/
#   medical_dataset_master_index.xlsx
#   prepared_medical_dataset_v3_outputs.zip
#
# This code:
# - downloads all available Kaggle datasets
# - resumes if interrupted
# - creates checkpoints per dataset/chunk
# - scans images safely
# - reads CSV metadata when available
# - infers labels/tasks/categories
# - separates real images from masks/annotation images
# - prevents data leakage using patient_id + exact_md5 + perceptual_hash
# - creates train/evaluation/testing/analysis manifests
# - creates a full Excel index with dataset paths, summaries, splits, labels, and manifest paths
# ============================================================

In [1]:
!pip -q install kagglehub pandas numpy pillow opencv-python tqdm openpyxl

import os
import re
import gc
import json
import cv2
import time
import shutil
import random
import zipfile
import hashlib
import kagglehub
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image, ImageOps
from tqdm import tqdm
from collections import Counter

# ============================================================
# CONFIG
# ============================================================


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_ROOT = Path("/kaggle/working/prepared_medical_dataset_v3")
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
MANIFESTS_DIR = OUTPUT_ROOT / "manifests"
REPORTS_DIR = OUTPUT_ROOT / "reports"
EXCEL_PATH = OUTPUT_ROOT / "medical_dataset_master_index.xlsx"

CHUNK_SIZE = 5000
RESUME = True

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

MIN_WIDTH = 64
MIN_HEIGHT = 64
SPLIT_RATIOS = {
    "training": 0.70,
    "evaluation": 0.10,
    "testing": 0.10,
    "analysis": 0.10,
}

LOW_CLASS_COUNT_THRESHOLD = 50

# For very large datasets, exact MD5 is expensive but useful for leakage.
# Keep True for strongest duplicate detection.
COMPUTE_EXACT_MD5 = True

# Perceptual hash detects near-duplicates.
COMPUTE_PHASH = True

# Do not copy images by default; manifests point to Kaggle input paths.
# This keeps output light and training-ready.
COPY_IMAGES = False

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# DATASETS LIST
# ============================================================


In [3]:
MEDICAL_DATASETS = [
    # Wounds / Injuries
    {"slug": "ibrahimfateen/wound-classification", "short_name": "wound_classification", "category": "wound", "task_hint": "wound_type"},
    {"slug": "yasinpratomo/wound-dataset", "short_name": "wound_dataset", "category": "wound", "task_hint": "wound_injury_type"},
    {"slug": "laithjj/diabetic-foot-ulcer-dfu", "short_name": "diabetic_foot_ulcer_dfu", "category": "wound", "task_hint": "diabetic_foot_ulcer"},
    {"slug": "khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes", "short_name": "dfu_4_classes", "category": "wound", "task_hint": "wound_stage"},
    {"slug": "sinemgokoz/pressure-ulcers-stages", "short_name": "pressure_ulcer_stages", "category": "wound", "task_hint": "pressure_ulcer_stage"},
    {"slug": "leoscode/wound-segmentation-images", "short_name": "wound_segmentation", "category": "wound", "task_hint": "wound_segmentation"},
    {"slug": "orvile/leprosy-chronic-wound-images-co2wounds-v2", "short_name": "leprosy_chronic_wounds", "category": "wound", "task_hint": "chronic_wound"},

    # Burns
    {"slug": "shubhambaid/skin-burn-dataset", "short_name": "skin_burn_yolo", "category": "burn", "task_hint": "burn_degree"},
    {"slug": "mohammaddimasnoufal/skin-burn-dataset", "short_name": "skin_burn_multiclass", "category": "burn", "task_hint": "burn_degree"},
    {"slug": "faresabbasai2022/burn-dataset13", "short_name": "burn_dataset13", "category": "burn", "task_hint": "burn_degree"},
    {"slug": "brendarangelolvera/human-skin-burns", "short_name": "human_skin_burns", "category": "burn", "task_hint": "burn_degree"},

    # Skin Cancer / Lesions
    {"slug": "kmader/skin-cancer-mnist-ham10000", "short_name": "ham10000", "category": "skin_cancer", "task_hint": "skin_cancer_type"},
    {"slug": "andrewmvd/isic-2019", "short_name": "isic_2019", "category": "skin_cancer", "task_hint": "skin_cancer_type"},
    {"slug": "salviohexia/isic-2019-skin-lesion-images-for-classification", "short_name": "isic_2019_classification", "category": "skin_cancer", "task_hint": "skin_cancer_type"},
    {"slug": "mahdavi1202/skin-cancer", "short_name": "pad_ufes_skin_cancer", "category": "skin_cancer", "task_hint": "skin_cancer_type"},
    {"slug": "abtahimajeed/pad-ufes-20", "short_name": "pad_ufes_20", "category": "skin_cancer", "task_hint": "skin_cancer_type"},
    {"slug": "nischaydnk/isic-2020-jpg-224x224-resized", "short_name": "isic_2020_224", "category": "skin_cancer", "task_hint": "benign_malignant"},
    {"slug": "nischaydnk/isic-2020-jpg-256x256-resized", "short_name": "isic_2020_256", "category": "skin_cancer", "task_hint": "benign_malignant"},
    {"slug": "riyaelizashaju/isic-skin-disease-image-dataset-labelled", "short_name": "isic_labelled_skin_disease", "category": "skin_cancer", "task_hint": "skin_cancer_type"},
    {"slug": "fanconic/skin-cancer-malignant-vs-benign", "short_name": "malignant_vs_benign", "category": "skin_cancer", "task_hint": "benign_malignant"},
    {"slug": "sani84/hiba-skin-lesion", "short_name": "hiba_skin_lesion", "category": "skin_cancer", "task_hint": "skin_cancer_type"},
    {"slug": "murtozalikhon/skin-cancer-classification", "short_name": "skin_cancer_classification", "category": "skin_cancer", "task_hint": "skin_cancer_type"},

    # Eczema / Psoriasis / Dermatitis
    {"slug": "ismailpromus/skin-diseases-image-dataset", "short_name": "skin_diseases_image_dataset", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
    {"slug": "adityush/eczema2", "short_name": "eczema2", "category": "eczema_dermatitis", "task_hint": "eczema"},
    {"slug": "sayedhossainjobayer/skin-diseases-dataset", "short_name": "skin_diseases_dataset_simple", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
    {"slug": "pallapurajkumar/psoriasis-skin-dataset", "short_name": "psoriasis_skin_dataset", "category": "psoriasis", "task_hint": "psoriasis"},
    {"slug": "adityaananda1/atomic-dermatitis", "short_name": "atopic_dermatitis", "category": "eczema_dermatitis", "task_hint": "atopic_dermatitis"},
    {"slug": "olcaybolat1/dermatology-dataset-classification", "short_name": "dermatology_dataset_classification", "category": "general_skin_disease", "task_hint": "general_skin_disease"},

    # General Skin Diseases
    {"slug": "shubhamgoel27/dermnet", "short_name": "dermnet", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
    {"slug": "pacificrm/skindiseasedataset", "short_name": "skin_disease_22", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
    {"slug": "youssefmohmmed/human-skin-diseases-image", "short_name": "human_skin_diseases", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
    {"slug": "haroonalam16/20-skin-diseases-dataset", "short_name": "twenty_skin_diseases", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
    {"slug": "aealemran/skin-disease-dataset-22-class", "short_name": "skin_disease_22_class_alt", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
    {"slug": "wariishayat/skin-disease-detection", "short_name": "skin_disease_detection", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
    {"slug": "sd20co001/image-dataset-for-skindiseases-dry-oily-normalskin", "short_name": "skin_diseases_plus_skin_type", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
    {"slug": "victor3452899385/dermnet-zhang", "short_name": "dermnet_zhang", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
    {"slug": "muhammadabdulsami/massive-skin-disease-balanced-dataset", "short_name": "massive_skin_disease_balanced", "category": "general_skin_disease", "task_hint": "general_skin_disease"},
]

# ============================================================
# LABEL MAPPING
# ============================================================

In [4]:
LABEL_PATTERNS = [
    # Burn
    ("burn_first_degree", ["first degree", "1st degree", "degree 1", "first_degree", "1st_degree", "superficial burn", "class 0", "burn 0"]),
    ("burn_second_degree", ["second degree", "2nd degree", "degree 2", "second_degree", "2nd_degree", "partial thickness", "class 1", "burn 1"]),
    ("burn_third_degree", ["third degree", "3rd degree", "degree 3", "third_degree", "3rd_degree", "full thickness", "class 2", "burn 2"]),
    ("no_burn", ["no burn", "no_burn", "normal burn", "no sunburn", "no_sunburn", "not burn"]),
    ("burn", ["burn", "burns", "skin burn", "sunburn"]),

    # Wound
    ("diabetic_foot_ulcer", ["diabetic foot ulcer", "diabetic_foot", "dfu", "diabetic wound", "diabetic ulcer"]),
    ("pressure_ulcer_stage_1", ["stage 1", "stage_1", "stage-i", "stage i"]),
    ("pressure_ulcer_stage_2", ["stage 2", "stage_2", "stage-ii", "stage ii"]),
    ("pressure_ulcer_stage_3", ["stage 3", "stage_3", "stage-iii", "stage iii"]),
    ("pressure_ulcer_stage_4", ["stage 4", "stage_4", "stage-iv", "stage iv"]),
    ("pressure_ulcer", ["pressure ulcer", "pressure_ulcer", "bedsore", "bed sore", "decubitus"]),
    ("surgical_wound", ["surgical wound", "surgical", "post operative", "postoperative"]),
    ("venous_ulcer", ["venous", "venous ulcer", "venous_ulcer"]),
    ("arterial_ulcer", ["arterial", "arterial ulcer", "arterial_ulcer"]),
    ("laceration", ["laceration", "lacerations"]),
    ("abrasion", ["abrasion", "abrasions"]),
    ("bruise", ["bruise", "bruises", "bruising"]),
    ("cut", ["cut", "cuts"]),
    ("stab_wound", ["stab", "stab wound", "stab_wound"]),
    ("injury", ["injury", "injuries", "trauma"]),
    ("wound", ["wound", "wounds", "ulcer", "ulcers"]),

    # Cancer / lesions
    ("melanoma", ["melanoma", "mel"]),
    ("melanocytic_nevus", ["melanocytic nevus", "nevus", "nevi", "nv"]),
    ("basal_cell_carcinoma", ["basal cell carcinoma", "bcc", "basal_cell"]),
    ("squamous_cell_carcinoma", ["squamous cell carcinoma", "scc", "squamous_cell"]),
    ("actinic_keratosis", ["actinic keratosis", "akiec", "actinic", "ak"]),
    ("benign_keratosis", ["benign keratosis", "bkl", "seborrheic keratosis", "seborrheic"]),
    ("dermatofibroma", ["dermatofibroma", "df"]),
    ("vascular_lesion", ["vascular lesion", "vasc", "vascular"]),
    ("benign", ["benign", "non-malignant", "non malignant"]),
    ("malignant", ["malignant", "cancer", "cancerous"]),

    # Eczema / dermatitis / psoriasis
    ("eczema", ["eczema", "eczemas"]),
    ("atopic_dermatitis", ["atopic dermatitis", "atopic", "atomic dermatitis", "dermatitis atopic"]),
    ("dermatitis", ["dermatitis", "contact dermatitis", "chronic dermatitis"]),
    ("psoriasis", ["psoriasis", "psoriatic"]),


    # General diseases
    ("acne_rosacea", ["acne and rosacea", "rosacea", "acne_rosacea"]),
    ("acne", ["acne", "pimple", "pimples"]),
    ("cellulitis", ["cellulitis"]),
    ("fungal_infection", ["fungal", "tinea", "ringworm", "athlete foot", "athlete_foot", "candida"]),
    ("warts", ["wart", "warts", "hpv"]),
    ("herpes", ["herpes", "shingles", "zoster"]),
    ("lupus", ["lupus"]),
    ("urticaria", ["urticaria", "hives"]),
    ("lichen_planus", ["lichen planus", "lichen_planus"]),
    ("vitiligo", ["vitiligo"]),
    ("normal", ["normal", "healthy", "clear", "no disease", "no_disease"]),
]

PATIENT_COLUMN_HINTS = ["patient", "patient_id", "patient id", "person", "subject", "subject_id", "participant", "volunteer", "client", "user", "case", "case_id"]
IMAGE_COLUMN_HINTS = ["image", "img", "file", "filename", "filepath", "path", "photo", "picture", "isic_id", "image_name"]
LABEL_COLUMN_HINTS = ["label", "class", "category", "condition", "diagnosis", "dx", "disease", "target", "benign_malignant", "lesion_type", "type", "stage", "degree"]



# ============================================================
# HELPERS
# ============================================================


In [5]:

def safe_name(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9_]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_") or "unknown"

def normalize_text(text):
    text = str(text).lower().replace("\\", "/")
    text = re.sub(r"[_\-.]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def list_images(root):
    root = Path(root)
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])

def list_csv_files(root):
    return sorted([p for p in Path(root).rglob("*.csv") if p.is_file()])

def file_md5(path, chunk_size=1024 * 1024):
    if not COMPUTE_EXACT_MD5:
        return ""
    md5 = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            md5.update(chunk)
    return md5.hexdigest()

def average_hash_pil(img, hash_size=16):
    if not COMPUTE_PHASH:
        return ""
    img = img.convert("L").resize((hash_size, hash_size), Image.Resampling.LANCZOS)
    arr = np.asarray(img, dtype=np.float32)
    avg = arr.mean()
    bits = arr > avg
    bit_string = "".join("1" if b else "0" for b in bits.flatten())
    width = int(np.ceil(len(bit_string) / 4))
    return f"{int(bit_string, 2):0{width}x}"

def infer_label_from_text(text):
    text_norm = normalize_text(text)
    for unified, patterns in LABEL_PATTERNS:
        for pat in patterns:
            p = normalize_text(pat)
            if len(p) <= 4:
                if re.search(rf"(^|[^a-z0-9]){re.escape(p)}([^a-z0-9]|$)", text_norm):
                    return pat, unified
            else:
                if p in text_norm:
                    return pat, unified
    return "unknown", "unknown"

def infer_main_task(category, task_hint, unified_label):
    if category == "burn":
        return "burn_degree_or_burn_detection"
    if category == "wound":
        if "stage" in unified_label or "degree" in unified_label:
            return "wound_stage_or_severity"
        if "segmentation" in task_hint:
            return "wound_segmentation"
        return "wound_type"
    if category == "skin_cancer":
        if unified_label in ["benign", "malignant"]:
            return "benign_malignant"
        return "skin_cancer_or_lesion_type"
    if category in ["eczema_dermatitis", "psoriasis"]:
        return "eczema_psoriasis_dermatitis"
    if category == "general_skin_disease":
        return "general_skin_disease"
    return task_hint or "unknown_task"

def infer_data_role(path_text):
    t = normalize_text(path_text)
    mask_words = ["mask", "masks", "segmentation", "ground truth", "ground_truth", "annotation", "annotations", "label mask"]
    if any(w in t for w in mask_words):
        return "mask_or_annotation_image"
    return "image"

def detect_columns(df):
    image_col = label_col = patient_col = None
    for col in df.columns:
        c = normalize_text(col)
        if any(h in c for h in IMAGE_COLUMN_HINTS):
            image_col = col
            break
    for col in df.columns:
        c = normalize_text(col)
        if any(h in c for h in LABEL_COLUMN_HINTS):
            label_col = col
            break
    for col in df.columns:
        c = normalize_text(col)
        if any(h in c for h in PATIENT_COLUMN_HINTS):
            patient_col = col
            break
    return image_col, label_col, patient_col

def build_csv_metadata_index(dataset_root):
    dataset_root = Path(dataset_root)
    csv_files = list_csv_files(dataset_root)
    index = {}

    for csv_path in csv_files:
        try:
            df = pd.read_csv(csv_path, low_memory=False)
        except Exception:
            continue

        if df.empty:
            continue

        image_col, label_col, patient_col = detect_columns(df)
        if image_col is None:
            continue

        for _, row in df.iterrows():
            image_value = str(row.get(image_col, "")).strip()
            if not image_value or image_value.lower() == "nan":
                continue

            p = Path(image_value)
            keys = {
                safe_name(str(p).replace("\\", "/")),
                safe_name(p.name),
                safe_name(p.stem),
                safe_name(str(image_value)),
            }

            raw_label = "unknown"
            unified_label = "unknown"

            if label_col is not None:
                raw_label = str(row.get(label_col, "unknown"))
                _, unified_label = infer_label_from_text(raw_label)

                col_norm = normalize_text(label_col)
                if col_norm == "target":
                    val = str(row.get(label_col, "")).strip().lower()
                    if val in ["1", "1.0", "true", "malignant"]:
                        raw_label = "malignant"
                        unified_label = "malignant"
                    elif val in ["0", "0.0", "false", "benign"]:
                        raw_label = "benign"
                        unified_label = "benign"

            patient_id = ""
            if patient_col is not None:
                patient_value = str(row.get(patient_col, "")).strip()
                if patient_value and patient_value.lower() != "nan":
                    patient_id = safe_name(patient_value)

            for key in keys:
                if key:
                    index[key] = {
                        "raw_label": raw_label,
                        "unified_label": unified_label,
                        "patient_id": patient_id,
                        "csv_source": str(csv_path),
                        "image_col": image_col,
                        "label_col": label_col if label_col else "",
                        "patient_col": patient_col if patient_col else ""
                    }

    return index

def lookup_metadata(image_path, dataset_root, metadata_index):
    image_path = Path(image_path)
    dataset_root = Path(dataset_root)
    rel = image_path.relative_to(dataset_root)

    candidates = [
        safe_name(str(rel).replace("\\", "/")),
        safe_name(image_path.name),
        safe_name(image_path.stem),
        safe_name(str(image_path)),
    ]

    for c in candidates:
        if c in metadata_index:
            return metadata_index[c]
    return None

def infer_patient_id_from_path(image_path, dataset_root, dataset_short_name):
    image_path = Path(image_path)
    dataset_root = Path(dataset_root)
    rel = image_path.relative_to(dataset_root)

    parts = [safe_name(p) for p in rel.parts[:-1]]
    stem = safe_name(image_path.stem)

    for part in reversed(parts):
        if re.search(r"(patient|person|subject|case|participant|client|user)[_0-9a-z]*", part):
            return f"{dataset_short_name}__{part}", "path_patient_folder"

    for part in reversed(parts):
        if re.fullmatch(r"(id_?)?\d{1,10}", part):
            return f"{dataset_short_name}__{part}", "path_numeric_folder"

    cleaned = stem
    cleaned = re.sub(r"(img|image|photo|picture|jpeg|jpg|png)", "", cleaned)
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")

    if cleaned:
        return f"{dataset_short_name}__{cleaned}", "filename_inferred"

    return f"{dataset_short_name}__{stem}", "filename_stem"

def analyze_image_quality(image_path):
    try:
        with Image.open(image_path) as img:
            img = ImageOps.exif_transpose(img)
            width, height = img.size
            mode = img.mode
            phash = average_hash_pil(img) if COMPUTE_PHASH else ""
            gray = np.array(img.convert("L"))
            brightness = float(np.mean(gray))
            blur_score = float(cv2.Laplacian(gray.astype(np.uint8), cv2.CV_64F).var())

        valid = True
        skip_reason = ""
        if width < MIN_WIDTH or height < MIN_HEIGHT:
            valid = False
            skip_reason = f"too_small_{width}x{height}"

        return {
            "width": int(width),
            "height": int(height),
            "mode": mode,
            "brightness": round(brightness, 4),
            "blur_score": round(blur_score, 4),
            "perceptual_hash": phash,
            "valid": valid,
            "skip_reason": skip_reason
        }

    except Exception as e:
        return {
            "width": 0,
            "height": 0,
            "mode": "unknown",
            "brightness": 0.0,
            "blur_score": 0.0,
            "perceptual_hash": "",
            "valid": False,
            "skip_reason": f"corrupt_or_unreadable: {str(e)}"
        }


# ============================================================
# DOWNLOAD + CHECKPOINTED SCAN
# ============================================================

def download_dataset(ds):
    slug = ds["slug"]
    try:
        path = kagglehub.dataset_download(slug)
        return {"status": "downloaded", "path": path, "error": ""}
    except Exception as e:
        return {"status": "failed", "path": "", "error": str(e)}

def get_completed_chunk_paths(dataset_checkpoint_dir):
    return sorted(dataset_checkpoint_dir.glob("chunk_*.csv"))

def read_existing_processed_paths(dataset_checkpoint_dir):
    processed = set()
    for chunk_file in get_completed_chunk_paths(dataset_checkpoint_dir):
        try:
            usecols = ["original_path"]
            temp = pd.read_csv(chunk_file, usecols=usecols)
            processed.update(temp["original_path"].astype(str).tolist())
        except Exception:
            continue
    return processed

def scan_one_dataset(ds, dataset_path):
    short_name = ds["short_name"]
    category = ds["category"]
    task_hint = ds["task_hint"]
    slug = ds["slug"]

    dataset_root = Path(dataset_path)
    dataset_checkpoint_dir = CHECKPOINT_DIR / short_name
    dataset_checkpoint_dir.mkdir(parents=True, exist_ok=True)

    final_manifest_path = dataset_checkpoint_dir / f"{short_name}_raw_manifest.csv"
    status_path = dataset_checkpoint_dir / "status.json"

    if RESUME and final_manifest_path.exists():
        print(f"RESUME: completed manifest exists for {short_name}")
        try:
            return pd.read_csv(final_manifest_path)
        except Exception:
            pass

    image_files = list_images(dataset_root)
    csv_files = list_csv_files(dataset_root)
    metadata_index = build_csv_metadata_index(dataset_root)

    processed_paths = read_existing_processed_paths(dataset_checkpoint_dir) if RESUME else set()
    remaining = [p for p in image_files if str(p) not in processed_paths]

    print("\n" + "=" * 80)
    print("Dataset:", short_name)
    print("Slug:", slug)
    print("Category:", category)
    print("Root:", dataset_root)
    print("Images found:", len(image_files))
    print("Already processed:", len(processed_paths))
    print("Remaining:", len(remaining))
    print("CSV files:", len(csv_files))
    print("Metadata index entries:", len(metadata_index))
    print("=" * 80)

    chunk_records = []
    chunk_id = len(get_completed_chunk_paths(dataset_checkpoint_dir))

    for image_path in tqdm(remaining, desc=f"Scanning {short_name}"):
        try:
            rel_path = image_path.relative_to(dataset_root)
            data_role = infer_data_role(str(rel_path))

            text_for_label = " ".join([
                str(rel_path),
                image_path.stem,
                " ".join(rel_path.parts),
                short_name,
                category,
                task_hint
            ])

            raw_label, unified_label = infer_label_from_text(text_for_label)

            patient_id, patient_source = infer_patient_id_from_path(
                image_path=image_path,
                dataset_root=dataset_root,
                dataset_short_name=short_name
            )

            metadata = lookup_metadata(image_path, dataset_root, metadata_index)

            if metadata:
                if metadata["unified_label"] != "unknown":
                    raw_label = metadata["raw_label"]
                    unified_label = metadata["unified_label"]

                if metadata["patient_id"]:
                    patient_id = f"{short_name}__{metadata['patient_id']}"
                    patient_source = "csv_patient_column"

            quality = analyze_image_quality(image_path)

            try:
                md5 = file_md5(image_path) if COMPUTE_EXACT_MD5 else ""
            except Exception:
                md5 = ""

            valid = quality["valid"]
            skip_reason = quality["skip_reason"]

            if unified_label == "unknown":
                valid = False
                skip_reason = "unknown_label"

            if data_role == "mask_or_annotation_image":
                # Keep in all_manifest, but exclude from classification-ready manifests.
                valid_for_classification = False
            else:
                valid_for_classification = bool(valid and unified_label != "unknown")

            main_task = infer_main_task(category, task_hint, unified_label)

            chunk_records.append({
                "dataset_slug": slug,
                "dataset_short_name": short_name,
                "category": category,
                "task_hint": task_hint,
                "main_task": main_task,
                "data_role": data_role,
                "dataset_root": str(dataset_root),
                "original_path": str(image_path),
                "prepared_path": str(image_path),
                "relative_path": str(rel_path),
                "filename": image_path.name,
                "stem": image_path.stem,
                "raw_label": raw_label,
                "unified_label": unified_label,
                "patient_id": patient_id,
                "patient_id_source": patient_source,
                "exact_md5": md5,
                "perceptual_hash": quality["perceptual_hash"],
                "width": quality["width"],
                "height": quality["height"],
                "mode": quality["mode"],
                "brightness": quality["brightness"],
                "blur_score": quality["blur_score"],
                "file_size_bytes": image_path.stat().st_size if image_path.exists() else 0,
                "valid": valid,
                "valid_for_classification": valid_for_classification,
                "skip_reason": skip_reason,
            })

        except Exception as e:
            chunk_records.append({
                "dataset_slug": slug,
                "dataset_short_name": short_name,
                "category": category,
                "task_hint": task_hint,
                "main_task": task_hint,
                "data_role": "error",
                "dataset_root": str(dataset_root),
                "original_path": str(image_path),
                "prepared_path": str(image_path),
                "relative_path": "",
                "filename": image_path.name,
                "stem": image_path.stem,
                "raw_label": "unknown",
                "unified_label": "unknown",
                "patient_id": f"{short_name}__error_{safe_name(image_path.stem)}",
                "patient_id_source": "error",
                "exact_md5": "",
                "perceptual_hash": "",
                "width": 0,
                "height": 0,
                "mode": "unknown",
                "brightness": 0,
                "blur_score": 0,
                "file_size_bytes": 0,
                "valid": False,
                "valid_for_classification": False,
                "skip_reason": f"scan_error: {str(e)}",
            })

        if len(chunk_records) >= CHUNK_SIZE:
            chunk_path = dataset_checkpoint_dir / f"chunk_{chunk_id:05d}.csv"
            pd.DataFrame(chunk_records).to_csv(chunk_path, index=False)
            chunk_records = []
            chunk_id += 1

            with open(status_path, "w", encoding="utf-8") as f:
                json.dump({
                    "short_name": short_name,
                    "images_found": len(image_files),
                    "processed_before_run": len(processed_paths),
                    "chunks_written": chunk_id,
                    "last_update": time.strftime("%Y-%m-%d %H:%M:%S")
                }, f, indent=2)

    if chunk_records:
        chunk_path = dataset_checkpoint_dir / f"chunk_{chunk_id:05d}.csv"
        pd.DataFrame(chunk_records).to_csv(chunk_path, index=False)
        chunk_records = []

    chunk_files = get_completed_chunk_paths(dataset_checkpoint_dir)

    if len(chunk_files) == 0:
        df = pd.DataFrame()
    else:
        df = pd.concat([pd.read_csv(p) for p in chunk_files], ignore_index=True)

    df.to_csv(final_manifest_path, index=False)

    with open(status_path, "w", encoding="utf-8") as f:
        json.dump({
            "short_name": short_name,
            "status": "completed",
            "images_found": len(image_files),
            "csv_files": len(csv_files),
            "metadata_entries": len(metadata_index),
            "rows_in_manifest": len(df),
            "last_update": time.strftime("%Y-%m-%d %H:%M:%S")
        }, f, indent=2)

    gc.collect()
    return df


# ============================================================
# STRONG LEAKAGE GROUPING
# ============================================================

class UnionFind:
    def __init__(self):
        self.parent = {}

    def add(self, x):
        if x not in self.parent:
            self.parent[x] = x

    def find(self, x):
        self.add(x)
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a, b):
        self.add(a)
        self.add(b)
        ra = self.find(a)
        rb = self.find(b)
        if ra != rb:
            self.parent[rb] = ra

def add_strong_leakage_groups(df):
    df = df.copy().reset_index(drop=True)
    uf = UnionFind()
    row_nodes = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building leakage groups"):
        row_node = f"row::{idx}"
        row_nodes.append(row_node)
        uf.add(row_node)

        patient_id = str(row.get("patient_id", "")).strip()
        exact_md5 = str(row.get("exact_md5", "")).strip()
        phash = str(row.get("perceptual_hash", "")).strip()

        if patient_id and patient_id.lower() not in ["nan", "unknown", "none"]:
            uf.union(row_node, f"patient::{patient_id}")
        if exact_md5 and exact_md5.lower() not in ["nan", "unknown", "none"]:
            uf.union(row_node, f"md5::{exact_md5}")
        if phash and phash.lower() not in ["nan", "unknown", "none"]:
            uf.union(row_node, f"phash::{phash}")

    df["strong_leakage_group"] = [uf.find(n) for n in row_nodes]
    return df


# ============================================================
# SPLIT
# ============================================================

def group_safe_split(valid_df):
    valid_df = valid_df.copy().reset_index(drop=True)

    groups = []
    for group_id, gdf in valid_df.groupby("strong_leakage_group"):
        label_counts = Counter(gdf["unified_label"].tolist())
        groups.append({
            "group": group_id,
            "count": len(gdf),
            "label_counts": label_counts,
            "major_label": label_counts.most_common(1)[0][0]
        })

    random.shuffle(groups)
    groups = sorted(groups, key=lambda x: x["count"], reverse=True)

    total_count = len(valid_df)
    target_counts = {split: int(round(total_count * ratio)) for split, ratio in SPLIT_RATIOS.items()}
    drift = total_count - sum(target_counts.values())
    target_counts["training"] += drift

    total_label_counts = Counter(valid_df["unified_label"].tolist())
    target_label_counts = {
        split: {label: total_label_counts[label] * SPLIT_RATIOS[split] for label in total_label_counts}
        for split in SPLIT_RATIOS
    }

    split_counts = {split: 0 for split in SPLIT_RATIOS}
    split_label_counts = {split: Counter() for split in SPLIT_RATIOS}
    group_to_split = {}

    for item in tqdm(groups, desc="Assigning groups to splits"):
        best_split = None
        best_score = None

        for split in SPLIT_RATIOS:
            size_after = split_counts[split] + item["count"]
            size_score = size_after / max(target_counts[split], 1)

            label_score = 0.0
            for label, count in item["label_counts"].items():
                after = split_label_counts[split][label] + count
                target = max(target_label_counts[split].get(label, 1.0), 1.0)
                label_score += after / target

            label_score = label_score / max(len(item["label_counts"]), 1)
            overflow_penalty = max(0.0, size_score - 1.0) * 4.0
            score = size_score + label_score + overflow_penalty

            if best_score is None or score < best_score:
                best_score = score
                best_split = split

        group_to_split[item["group"]] = best_split
        split_counts[best_split] += item["count"]
        split_label_counts[best_split].update(item["label_counts"])

    valid_df["split"] = valid_df["strong_leakage_group"].map(group_to_split)
    return valid_df

def leakage_check(df):
    report = {
        "status": "PASS",
        "issues": [],
        "split_counts": df["split"].value_counts().to_dict(),
        "overlaps": {}
    }

    split_names = sorted(df["split"].dropna().unique().tolist())
    check_columns = ["strong_leakage_group", "patient_id", "exact_md5", "perceptual_hash"]

    for col in check_columns:
        col_overlaps = {}
        split_sets = {
            split: set(df[df["split"] == split][col].dropna().astype(str))
            for split in split_names
        }

        for i in range(len(split_names)):
            for j in range(i + 1, len(split_names)):
                a = split_names[i]
                b = split_names[j]
                inter = split_sets[a].intersection(split_sets[b])
                inter = {x for x in inter if x and x.lower() not in ["nan", "unknown", "none"]}
                if inter:
                    col_overlaps[f"{a}_vs_{b}"] = list(sorted(inter))[:100]

        report["overlaps"][col] = col_overlaps
        if col_overlaps:
            report["status"] = "FAIL"
            report["issues"].append(f"Overlap detected in {col}")

    return report


# ============================================================
# EXCEL EXPORT
# ============================================================

def write_excel_report(
    excel_path,
    download_status,
    dataset_summary,
    label_summary,
    split_summary,
    task_summary,
    category_summary,
    manifest_index,
    leakage_report,
    skipped_summary,
    sample_manifest
):
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        readme = pd.DataFrame({
            "Item": [
                "Purpose",
                "Main training input",
                "Do not re-split",
                "Leakage status",
                "All manifest path",
                "Training manifest path",
                "Evaluation manifest path",
                "Testing manifest path",
                "Analysis manifest path",
                "Note"
            ],
            "Value": [
                "Medical / Treatment Mode dataset index and manifests",
                str(MANIFESTS_DIR / "classification_training_manifest.csv"),
                "YES - use fixed manifests only",
                leakage_report.get("status", "UNKNOWN"),
                str(MANIFESTS_DIR / "classification_all_manifest.csv"),
                str(MANIFESTS_DIR / "classification_training_manifest.csv"),
                str(MANIFESTS_DIR / "classification_evaluation_manifest.csv"),
                str(MANIFESTS_DIR / "classification_testing_manifest.csv"),
                str(MANIFESTS_DIR / "classification_analysis_manifest.csv"),
                "Excel is an index/report. Use CSV manifests for training because they are faster and safer."
            ]
        })
        readme.to_excel(writer, sheet_name="README", index=False)
        download_status.to_excel(writer, sheet_name="Dataset_Paths", index=False)
        manifest_index.to_excel(writer, sheet_name="Manifest_Index", index=False)
        dataset_summary.to_excel(writer, sheet_name="Dataset_Summary", index=False)
        label_summary.to_excel(writer, sheet_name="Label_Summary", index=False)
        split_summary.to_excel(writer, sheet_name="Split_Summary", index=False)
        task_summary.to_excel(writer, sheet_name="Task_Summary", index=False)
        category_summary.to_excel(writer, sheet_name="Category_Summary", index=False)
        skipped_summary.to_excel(writer, sheet_name="Skipped_Summary", index=False)

        leak_df = pd.DataFrame({
            "key": ["status", "issues", "split_counts"],
            "value": [
                leakage_report.get("status"),
                json.dumps(leakage_report.get("issues", []), ensure_ascii=False),
                json.dumps(leakage_report.get("split_counts", {}), ensure_ascii=False),
            ]
        })
        leak_df.to_excel(writer, sheet_name="Leakage_Report", index=False)

        sample_manifest.head(5000).to_excel(writer, sheet_name="Manifest_Sample_5000", index=False)

    # Basic formatting
    from openpyxl import load_workbook
    wb = load_workbook(excel_path)

    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        for col in ws.columns:
            max_len = 0
            col_letter = col[0].column_letter
            for cell in col[:200]:
                try:
                    max_len = max(max_len, len(str(cell.value)) if cell.value is not None else 0)
                except Exception:
                    pass
            ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 55)

    wb.save(excel_path)


# ============================================================
# MAIN RUN
# ============================================================

download_records = []
dataset_dfs = []

for ds in MEDICAL_DATASETS:
    result = download_dataset(ds)

    download_records.append({
        "slug": ds["slug"],
        "short_name": ds["short_name"],
        "category": ds["category"],
        "task_hint": ds["task_hint"],
        "status": result["status"],
        "dataset_path": result["path"],
        "error": result["error"],
        "checkpoint_folder": str(CHECKPOINT_DIR / ds["short_name"]),
        "dataset_manifest": str(CHECKPOINT_DIR / ds["short_name"] / f"{ds['short_name']}_raw_manifest.csv")
    })

    if result["status"] == "downloaded":
        try:
            df_ds = scan_one_dataset(ds, result["path"])
            if len(df_ds) > 0:
                dataset_dfs.append(df_ds)
        except Exception as e:
            print("SCAN FAILED:", ds["short_name"], str(e))
            download_records[-1]["status"] = "scan_failed"
            download_records[-1]["error"] = str(e)

    # Update download status after every dataset
    pd.DataFrame(download_records).to_csv(REPORTS_DIR / "download_status_live.csv", index=False)

if len(dataset_dfs) == 0:
    raise RuntimeError("No dataset manifests were created.")

raw_df = pd.concat(dataset_dfs, ignore_index=True)
raw_df.to_csv(MANIFESTS_DIR / "raw_full_manifest.csv", index=False)

download_status = pd.DataFrame(download_records)

# Classification-ready data: real image, valid label, not mask
classification_df = raw_df[
    (raw_df["valid_for_classification"] == True) &
    (raw_df["data_role"] == "image") &
    (raw_df["unified_label"] != "unknown")
].copy()

mask_df = raw_df[raw_df["data_role"] != "image"].copy()
skipped_df = raw_df[
    (raw_df["valid_for_classification"] != True) |
    (raw_df["unified_label"] == "unknown")
].copy()

mask_df.to_csv(MANIFESTS_DIR / "segmentation_or_mask_related_manifest.csv", index=False)
skipped_df.to_csv(MANIFESTS_DIR / "skipped_or_not_classification_ready.csv", index=False)

if len(classification_df) == 0:
    raise RuntimeError("No classification-ready images after filtering.")

classification_df = add_strong_leakage_groups(classification_df)
split_df = group_safe_split(classification_df)

leak_report = leakage_check(split_df)

# Save main CSV manifests
split_df.to_csv(MANIFESTS_DIR / "classification_all_manifest.csv", index=False)

for split in SPLIT_RATIOS:
    part = split_df[split_df["split"] == split].copy()
    part.to_csv(MANIFESTS_DIR / f"classification_{split}_manifest.csv", index=False)

# Also save generic names for future training code
split_df.to_csv(MANIFESTS_DIR / "all_manifest.csv", index=False)
split_df[split_df["split"] == "training"].to_csv(MANIFESTS_DIR / "training_manifest.csv", index=False)
split_df[split_df["split"] == "evaluation"].to_csv(MANIFESTS_DIR / "evaluation_manifest.csv", index=False)
split_df[split_df["split"] == "testing"].to_csv(MANIFESTS_DIR / "testing_manifest.csv", index=False)
split_df[split_df["split"] == "analysis"].to_csv(MANIFESTS_DIR / "analysis_manifest.csv", index=False)

# Save by task
by_task_dir = MANIFESTS_DIR / "by_task"
by_task_dir.mkdir(exist_ok=True)

for task in sorted(split_df["main_task"].dropna().unique()):
    task_df = split_df[split_df["main_task"] == task].copy()
    task_df.to_csv(by_task_dir / f"{safe_name(task)}_all.csv", index=False)

    for split in SPLIT_RATIOS:
        task_df[task_df["split"] == split].to_csv(by_task_dir / f"{safe_name(task)}_{split}.csv", index=False)

# Save by category
by_category_dir = MANIFESTS_DIR / "by_category"
by_category_dir.mkdir(exist_ok=True)

for cat in sorted(split_df["category"].dropna().unique()):
    cat_df = split_df[split_df["category"] == cat].copy()
    cat_df.to_csv(by_category_dir / f"{safe_name(cat)}_all.csv", index=False)

    for split in SPLIT_RATIOS:
        cat_df[cat_df["split"] == split].to_csv(by_category_dir / f"{safe_name(cat)}_{split}.csv", index=False)

# Reports
dataset_summary = (
    raw_df.groupby(["dataset_short_name", "dataset_slug", "category", "task_hint"])
    .agg(
        total_images=("original_path", "count"),
        classification_ready=("valid_for_classification", "sum"),
        mask_or_annotation_images=("data_role", lambda x: int((x != "image").sum())),
        unknown_label=("unified_label", lambda x: int((x == "unknown").sum())),
        patient_groups=("patient_id", "nunique"),
        dataset_root=("dataset_root", "first"),
    )
    .reset_index()
    .sort_values(["category", "dataset_short_name"])
)

label_summary = (
    split_df.groupby(["main_task", "unified_label"])
    .agg(
        total=("prepared_path", "count"),
        groups=("strong_leakage_group", "nunique"),
        datasets=("dataset_short_name", lambda x: ", ".join(sorted(set(x))[:8])),
        categories=("category", lambda x: ", ".join(sorted(set(x)))),
    )
    .reset_index()
    .sort_values(["main_task", "unified_label"])
)

label_summary["overfitting_risk"] = np.where(
    label_summary["total"] < LOW_CLASS_COUNT_THRESHOLD,
    "HIGH_RISK_LOW_COUNT",
    "OK"
)

split_summary = (
    split_df.groupby(["main_task", "split", "unified_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["main_task", "split", "unified_label"])
)

task_summary = (
    split_df.groupby(["main_task", "split"])
    .size()
    .reset_index(name="count")
    .sort_values(["main_task", "split"])
)

category_summary = (
    split_df.groupby(["category", "split"])
    .size()
    .reset_index(name="count")
    .sort_values(["category", "split"])
)

skipped_summary = (
    skipped_df.groupby(["dataset_short_name", "skip_reason"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_short_name", "count"], ascending=[True, False])
)

duplicate_summary = (
    split_df.groupby("perceptual_hash")
    .agg(
        count=("perceptual_hash", "count"),
        labels=("unified_label", lambda x: ", ".join(sorted(set(x))[:10])),
        datasets=("dataset_short_name", lambda x: ", ".join(sorted(set(x))[:10])),
        splits=("split", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)
duplicate_summary = duplicate_summary[duplicate_summary["count"] > 1]

manifest_index = pd.DataFrame([
    {"name": "classification_all", "path": str(MANIFESTS_DIR / "classification_all_manifest.csv"), "use": "all classification-ready data with fixed split column"},
    {"name": "training", "path": str(MANIFESTS_DIR / "classification_training_manifest.csv"), "use": "training only"},
    {"name": "evaluation", "path": str(MANIFESTS_DIR / "classification_evaluation_manifest.csv"), "use": "validation/model selection"},
    {"name": "testing", "path": str(MANIFESTS_DIR / "classification_testing_manifest.csv"), "use": "final test only"},
    {"name": "analysis", "path": str(MANIFESTS_DIR / "classification_analysis_manifest.csv"), "use": "manual analysis/demo inspection"},
    {"name": "mask_related", "path": str(MANIFESTS_DIR / "segmentation_or_mask_related_manifest.csv"), "use": "segmentation/mask related files"},
    {"name": "skipped", "path": str(MANIFESTS_DIR / "skipped_or_not_classification_ready.csv"), "use": "not ready for classification training"},
    {"name": "by_task_folder", "path": str(by_task_dir), "use": "task-specific manifests"},
    {"name": "by_category_folder", "path": str(by_category_dir), "use": "category-specific manifests"},
])

# Save report files
download_status.to_csv(REPORTS_DIR / "download_status.csv", index=False)
dataset_summary.to_csv(REPORTS_DIR / "dataset_summary.csv", index=False)
label_summary.to_csv(REPORTS_DIR / "label_summary.csv", index=False)
split_summary.to_csv(REPORTS_DIR / "split_summary.csv", index=False)
task_summary.to_csv(REPORTS_DIR / "task_summary.csv", index=False)
category_summary.to_csv(REPORTS_DIR / "category_summary.csv", index=False)
skipped_summary.to_csv(REPORTS_DIR / "skipped_summary.csv", index=False)
duplicate_summary.to_csv(REPORTS_DIR / "duplicate_groups.csv", index=False)
manifest_index.to_csv(REPORTS_DIR / "manifest_index.csv", index=False)

with open(REPORTS_DIR / "leakage_report.json", "w", encoding="utf-8") as f:
    json.dump(leak_report, f, indent=2, ensure_ascii=False)

data_config = {
    "dataset_name": "prepared_medical_treatment_dataset_v3",
    "purpose": "Input-ready manifests for future medical AI models",
    "excel_master_index": str(EXCEL_PATH),
    "image_col": "prepared_path",
    "label_col": "unified_label",
    "task_col": "main_task",
    "category_col": "category",
    "split_col": "split",
    "group_col": "strong_leakage_group",
    "patient_col": "patient_id",
    "leakage_status": leak_report["status"],
    "manifests": {
        "all": str(MANIFESTS_DIR / "classification_all_manifest.csv"),
        "training": str(MANIFESTS_DIR / "classification_training_manifest.csv"),
        "evaluation": str(MANIFESTS_DIR / "classification_evaluation_manifest.csv"),
        "testing": str(MANIFESTS_DIR / "classification_testing_manifest.csv"),
        "analysis": str(MANIFESTS_DIR / "classification_analysis_manifest.csv"),
        "by_task_folder": str(by_task_dir),
        "by_category_folder": str(by_category_dir),
        "mask_related": str(MANIFESTS_DIR / "segmentation_or_mask_related_manifest.csv"),
    },
    "rules": [
        "Do not re-split in training code.",
        "Use training manifest only for training.",
        "Use evaluation manifest for model selection.",
        "Use testing manifest only once for final testing.",
        "Use analysis manifest for visual/manual inspection.",
        "Excel file is an index and report; use CSV manifests for model training."
    ]
}

with open(REPORTS_DIR / "data_config.json", "w", encoding="utf-8") as f:
    json.dump(data_config, f, indent=2, ensure_ascii=False)

# Excel master index
write_excel_report(
    excel_path=EXCEL_PATH,
    download_status=download_status,
    dataset_summary=dataset_summary,
    label_summary=label_summary,
    split_summary=split_summary,
    task_summary=task_summary,
    category_summary=category_summary,
    manifest_index=manifest_index,
    leakage_report=leak_report,
    skipped_summary=skipped_summary,
    sample_manifest=split_df
)

# Zip outputs, excluding huge checkpoint chunks to keep zip lighter
zip_path = Path("/kaggle/working/prepared_medical_dataset_v3_outputs.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_ROOT.rglob("*"):
        if not p.is_file():
            continue

        # Keep reports/manifests/excel/config. Skip chunk checkpoints from zip.
        rel = p.relative_to(OUTPUT_ROOT)
        if str(rel).startswith("checkpoints/") and p.suffix.lower() == ".csv":
            continue

        z.write(p, p.relative_to(OUTPUT_ROOT.parent))

print("\n" + "=" * 90)
print("MEDICAL DATASET V3 PIPELINE FINISHED")
print("=" * 90)
print("Output folder:", OUTPUT_ROOT)
print("Excel master index:", EXCEL_PATH)
print("ZIP outputs:", zip_path)
print("Leakage status:", leak_report["status"])
print("Raw images scanned:", len(raw_df))
print("Classification-ready images:", len(split_df))
print("Skipped/not classification-ready:", len(skipped_df))
print("Mask/annotation-related images:", len(mask_df))

print("\nMain training inputs:")
print("Training:", MANIFESTS_DIR / "classification_training_manifest.csv")
print("Evaluation:", MANIFESTS_DIR / "classification_evaluation_manifest.csv")
print("Testing:", MANIFESTS_DIR / "classification_testing_manifest.csv")
print("Analysis:", MANIFESTS_DIR / "classification_analysis_manifest.csv")
print("Config:", REPORTS_DIR / "data_config.json")

print("\nExcel sheets include:")
print("README, Dataset_Paths, Manifest_Index, Dataset_Summary, Label_Summary, Split_Summary, Task_Summary, Category_Summary, Skipped_Summary, Leakage_Report, Manifest_Sample_5000")

print("\nDataset summary:")
display(dataset_summary)

print("\nLabel summary:")
display(label_summary)

print("\nSplit summary:")
display(split_summary)

print("\nManifest index:")
display(manifest_index)


Dataset: wound_classification
Slug: ibrahimfateen/wound-classification
Category: wound
Root: /kaggle/input/datasets/ibrahimfateen/wound-classification
Images found: 2940
Already processed: 0
Remaining: 2940
CSV files: 0
Metadata index entries: 0


Scanning wound_classification: 100%|██████████| 2940/2940 [00:41<00:00, 70.91it/s]



Dataset: wound_dataset
Slug: yasinpratomo/wound-dataset
Category: wound
Root: /kaggle/input/datasets/yasinpratomo/wound-dataset
Images found: 431
Already processed: 0
Remaining: 431
CSV files: 0
Metadata index entries: 0


Scanning wound_dataset: 100%|██████████| 431/431 [00:05<00:00, 84.86it/s]



Dataset: diabetic_foot_ulcer_dfu
Slug: laithjj/diabetic-foot-ulcer-dfu
Category: wound
Root: /kaggle/input/datasets/laithjj/diabetic-foot-ulcer-dfu
Images found: 2671
Already processed: 0
Remaining: 2671
CSV files: 0
Metadata index entries: 0


...

,main_task,unified_label,total,groups,datasets,categories,overfitting_risk
0,benign_malignant,benign,72846,49353,"isic_2020_224, isic_2020_256, malignant_vs_benign",skin_cancer,OK
1,benign_malignant,malignant,31543,26998,"hiba_skin_lesion, isic_2019, pad_ufes_20, pad_...",skin_cancer,OK
2,burn_degree_or_burn_detection,burn,2319,1941,"burn_dataset13, human_skin_burns, skin_burn_yolo",burn,OK
3,burn_degree_or_burn_detection,burn_first_degree,3646,2301,"burn_dataset13, human_skin_burns, skin_burn_mu...",burn,OK
4,burn_degree_or_burn_detection,burn_second_degree,1719,1356,"burn_dataset13, human_skin_burns, skin_burn_mu...",burn,OK
...,...,...,...,...,...,...,...
62,wound_type,pressure_ulcer,381,380,"pressure_ulcer_stages, wound_classification",wound,OK
63,wound_type,stab_wound,23,22,wound_dataset,wound,HIGH_RISK_LOW_COUNT
64,wound_type,surgical_wound,420,418,wound_classification,wound,OK
65,wound_type,venous_ulcer,494,492,wound_classification,wound,OK



Split summary:


,main_task,split,unified_label,count
0,benign_malignant,analysis,benign,7278
1,benign_malignant,analysis,malignant,3132
2,benign_malignant,evaluation,benign,7280
3,benign_malignant,evaluation,malignant,3100
4,benign_malignant,testing,benign,7277
...,...,...,...,...
248,wound_type,training,pressure_ulcer,267
249,wound_type,training,stab_wound,17
250,wound_type,training,surgical_wound,294
251,wound_type,training,venous_ulcer,318



Manifest index:


,name,path,use
0,classification_all,/kaggle/working/prepared_medical_dataset_v3/ma...,all classification-ready data with fixed split...
1,training,/kaggle/working/prepared_medical_dataset_v3/ma...,training only
2,evaluation,/kaggle/working/prepared_medical_dataset_v3/ma...,validation/model selection
3,testing,/kaggle/working/prepared_medical_dataset_v3/ma...,final test only
4,analysis,/kaggle/working/prepared_medical_dataset_v3/ma...,manual analysis/demo inspection
5,mask_related,/kaggle/working/prepared_medical_dataset_v3/ma...,segmentation/mask related files
6,skipped,/kaggle/working/prepared_medical_dataset_v3/ma...,not ready for classification training
7,by_task_folder,/kaggle/working/prepared_medical_dataset_v3/ma...,task-specific manifests
8,by_category_folder,/kaggle/working/prepared_medical_dataset_v3/ma...,category-specific manifests
